# Chapter 4. 동적계획법 — 실습 노트북

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml2/chapter05_policy_evaluation.ipynb)

책 본문: [Chapter 4](https://smhanlab.com/book-ml/kor/ml2/chapter04.html)

이 노트북은 책의 반복적 정책평가(iterative policy evaluation) 함수를
그대로 실행하고, 손으로 계산한 값과 맞는지 직접 확인합니다.

## 1. 장난감 MDP 만들기

연습문제 2와 똑같은 3-state MDP를 씁니다:
State 0 → State 1(보상 -1) → State 2(보상 -1, **터미널**), 할인율 \(\gamma=0.9\).

터미널 상태는 자기 자신으로만 전이하고 보상 0을 주는 self-loop로
표현합니다 — 그래야 \(V(2)=0\)이 벨만방정식의 고정점으로 자연스럽게
나옵니다.

In [ ]:
# P[s][a] = [(prob, next_state), ...],  R[s][a] = 즉시 보상
P = [
    [[(1.0, 1)]],   # state 0 --(action 0)--> state 1
    [[(1.0, 2)]],   # state 1 --(action 0)--> state 2
    [[(1.0, 2)]],   # state 2 (terminal) --(action 0)--> state 2 (self-loop)
]
R = [
    [-1],  # state 0
    [-1],  # state 1
    [0],   # state 2 (terminal)
]
policy = [0, 0, 0]  # 각 상태에서 항상 action 0
gamma = 0.9

## 2. 반복적 정책평가 (책 5.4절 코드 그대로)

In [ ]:
def policy_evaluation(P, R, policy, gamma, theta=1e-6):
    n_states = len(P)
    V = [0.0] * n_states
    while True:
        delta = 0
        for s in range(n_states):
            a = policy[s]
            v_new = R[s][a] + gamma * sum(prob * V[next_s] for prob, next_s in P[s][a])
            delta = max(delta, abs(v_new - V[s]))
            V[s] = v_new
        if delta < theta:
            break
    return V

V = policy_evaluation(P, R, policy, gamma)
for s, v in enumerate(V):
    print(f"V({s}) = {v:.4f}")

## 3. 손으로 계산한 값과 비교 (연습문제 2)

터미널부터 거꾸로 대입(backward induction)하면:

\\(V(2) = 0\\)
\\(V(1) = -1 + 0.9 \times V(2) = -1\\)
\\(V(0) = -1 + 0.9 \times V(1) = -1.9\\)

코드 결과와 정확히 맞는지 확인합니다.

In [ ]:
hand_calc = {2: 0.0, 1: -1.0, 0: -1.9}
for s in range(3):
    assert abs(V[s] - hand_calc[s]) < 1e-4, f"state {s} mismatch: {V[s]} vs {hand_calc[s]}"
print("모두 일치! 반복 계산이 손으로 푼 값과 정확히 같습니다.")

## 4. 벨만방정식이 실제로 성립하는지 직접 검증

\\(V^\pi(s) = R(s,\pi(s)) + \gamma \sum_{s'} P(s'|s,\pi(s)) V^\pi(s')\\)
가 계산된 V에서 정말 등호로 성립하는지 각 상태마다 재계산해서 비교합니다.

In [ ]:
for s in range(3):
    a = policy[s]
    rhs = R[s][a] + gamma * sum(prob * V[next_s] for prob, next_s in P[s][a])
    print(f"state {s}: V(s)={V[s]:.4f}  Bellman 우변={rhs:.4f}  일치={abs(V[s]-rhs) < 1e-6}")